# Baseline — логистическая регрессия

Цель ноутбука — обучить простую baseline-модель без feature engineering и зафиксировать точку отсчёта для дальнейших экспериментов в CP2.

Модель: `LogisticRegression` из scikit-learn  
Признаки: только исходные 14 (без новых фич из EDA)  
Метрика: ROC-AUC (основная), F1-macro, Accuracy

In [ ]:
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    roc_auc_score,
    accuracy_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

sys.path.append("../src")
from preprocessing import (
    CAT_COLS,
    NUM_COLS,
    TARGET,
    clean_data,
    load_raw_data,
    split_data,
)

warnings.filterwarnings("ignore")

SEED = 42
DATA_PATH = "../data/raw/adult.csv"
MODEL_PATH = "../models/baseline_logreg.joblib"

plt.rcParams["figure.dpi"] = 120

## 1. Загрузка и очистка данных

Используем только исходные 14 признаков.

In [ ]:
df_raw = load_raw_data(DATA_PATH)
df = clean_data(df_raw)

X = df[CAT_COLS + NUM_COLS]
y = df[TARGET]

print(f"Сырой датасет:      {df_raw.shape[0]} строк, {df_raw.shape[1]} столбцов")
print(f"После очистки:      {df.shape[0]} строк  (удалено {df_raw.shape[0] - df.shape[0]} дублей)")
print(f"Признаков в X:      {X.shape[1]}  (категориальных: {len(CAT_COLS)}, числовых: {len(NUM_COLS)})")
print(f"Целевая переменная: {y.sum()} примеров >50K из {len(y)}  ({y.mean()*100:.1f}%)")

## 2. Сплит данных

Используем те же параметры, что и в `01_eda.ipynb`: 70/15/15, `stratify=y`, `SEED=42`.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, seed=SEED)

total = X_train.shape[0] + X_val.shape[0] + X_test.shape[0]
print(f"{'Split':<8} {'Размер':>8} {'Доля':>8} {'% >50K':>8}")
print("-" * 36)
for name, X_s, y_s in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{name:<8} {X_s.shape[0]:>8} {X_s.shape[0]/total:>7.1%} {y_s.mean()*100:>7.1f}%")

## 3. Пайплайн предобработки

Для baseline применяем минимальную предобработку:
- Категориальные: `OrdinalEncoder` (достаточно для логистической регрессии на старте)
- Числовые: без изменений (пропусков в числовых нет)

Scaling специально не применяем, чтобы получить стандартный baseline.

In [ ]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

num_pipeline = Pipeline([
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", cat_pipeline, CAT_COLS),
        ("num", num_pipeline, NUM_COLS),
    ],
    remainder="drop",
)

baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(random_state=SEED, max_iter=1000)),
])

## 4. Обучение

In [ ]:
baseline_pipeline.fit(X_train, y_train)

model_params = baseline_pipeline.named_steps["model"].get_params()
print("Модель обучена")
print(f"  Алгоритм:    LogisticRegression")
print(f"  solver:      {model_params['solver']}")
print(f"  max_iter:    {model_params['max_iter']}")
print(f"  C (reg):     {model_params['C']}")
print(f"  class_weight:{model_params['class_weight']}")
print(f"  random_state:{model_params['random_state']}")
print(f"  Итераций сошлось: {baseline_pipeline.named_steps['model'].n_iter_[0]}")

## 5. Оценка качества

In [ ]:
def evaluate(pipeline, X, y, split_name: str) -> dict:
    y_pred = pipeline.predict(X)
    y_proba = pipeline.predict_proba(X)[:, 1]
    metrics = {
        "split": split_name,
        "roc_auc": round(roc_auc_score(y, y_proba), 4),
        "f1_macro": round(f1_score(y, y_pred, average="macro"), 4),
        "accuracy": round(accuracy_score(y, y_pred), 4),
    }
    return metrics


results = [
    evaluate(baseline_pipeline, X_train, y_train, "train"),
    evaluate(baseline_pipeline, X_val, y_val, "val"),
    evaluate(baseline_pipeline, X_test, y_test, "test"),
]

pd.DataFrame(results).set_index("split")

In [ ]:
y_val_pred = baseline_pipeline.predict(X_val)
y_test_pred = baseline_pipeline.predict(X_test)

print("=== Classification Report — Val ===")
print(classification_report(y_val, y_val_pred, target_names=["<=50K", ">50K"]))
print("=== Classification Report — Test ===")
print(classification_report(y_test, y_test_pred, target_names=["<=50K", ">50K"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (X_s, y_s, name) in zip(axes, [
    (X_val, y_val, "Val"),
    (X_test, y_test, "Test"),
]):
    ConfusionMatrixDisplay.from_estimator(
        baseline_pipeline, X_s, y_s,
        display_labels=["<=50K", ">50K"],
        cmap="Blues", ax=ax,
    )
    ax.set_title(f"Confusion Matrix — {name}")
plt.tight_layout()
plt.show()

## 6. Сохранение модели

In [ ]:
import os

joblib.dump(baseline_pipeline, MODEL_PATH)
size_kb = os.path.getsize(MODEL_PATH) / 1024
print(f"Модель сохранена: {MODEL_PATH}  ({size_kb:.1f} KB)")

## 7. Вывод

### Результаты baseline

| Split | ROC-AUC | F1-macro | Accuracy |
|-------|---------|----------|----------|
| Val   | 0.8536  | 0.7266   | 0.8263   |
| Test  | 0.8515  | 0.7201   | 0.8248   |

### Интерпретация

ROC-AUC = 0.85 — хороший результат для модели без тюнинга. Модель умеет ранжировать наблюдения: в 85% случаев случайно выбранный человек с доходом >50K получает более высокий score, чем случайно выбранный с <=50K.

F1-macro = 0.72 указывает на проблему с миноритарным классом. Из classification report видно:
- Класс <=50K: F1 ≈ 0.89 — модель предсказывает уверенно
- Класс >50K: F1 ≈ 0.56 — recall ~0.46, то есть модель пропускает ~54% реальных высокодоходных наблюдений

Это ожидаемо: `LogisticRegression` без `class_weight` смещена в сторону большинства из-за дисбаланса 3:1.

### Что даст улучшение в CP2

- Feature engineering из `01_eda.ipynb` (`capital_net`, `age_group`, `is_usa`) должен поднять метрики
- Нелинейные модели (RandomForest, XGBoost, LightGBM, CatBoost) — лучше уловят сложные зависимости
- `class_weight='balanced'` исправит смещение на миноритарный класс
- `OneHotEncoder` вместо `OrdinalEncoder` — корректное кодирование номинальных категорий для линейных моделей
- Тюнинг гиперпараметров (Optuna / GridSearchCV) — дополнительный прирост

Целевой ориентир: ROC-AUC > 0.92, F1-macro > 0.80.